# 오토인코더로 이미지의 특징을 추출하기

In [ ]:
import torch
import torchvision
import torch.nn.functional as F
from torch import nn, optim
from torchvision import transforms, datasets

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import numpy as np
import os

In [ ]:
# 하이퍼파라미터 설정
EPOCH = 10 # 학습 반복 횟수 설정
BATCH_SIZE = 64 # 한 번에 학습할 데이터 묶음 크기 설정
USE_CUDA = torch.cuda.is_available() # GPU 사용 가능 여부 확인
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 학습 기기 설정
print("Using Device:", DEVICE) # 현재 사용 중인 기기 출력

In [ ]:
# Fashion MNIST 데이터셋 로드
trainset = datasets.FashionMNIST( # Fashion MNIST 학습 데이터셋 로드
    root      = './.data/', # 데이터 저장 경로
    train     = True, # 학습용 데이터 선택
    download  = True, # 데이터가 없으면 다운로드
    transform = transforms.ToTensor() # 이미지를 텐서로 변환
)
train_loader = torch.utils.data.DataLoader( # 데이터를 배치 단위로 불러오는 도구
    dataset     = trainset, # 로드한 데이터셋 지정
    batch_size  = BATCH_SIZE, # 배치 크기 지정
    shuffle     = True, # 데이터를 무작위로 섞음
    num_workers = 0 # 병렬 처리 프로세스 수 설정
)

In [ ]:
# 오토인코더 모델 정의
class Autoencoder(nn.Module): # 오토인코더 모델 클래스 정의
    def __init__(self): # 초기화 함수
        super(Autoencoder, self).__init__() # 부모 클래스 초기화

        self.encoder = nn.Sequential( # 인코더 부분 정의
            nn.Linear(28*28, 128), # 입력 이미지를 128차원으로 선형 변환
            nn.ReLU(), # 비선형 활성화 함수 적용
            nn.Linear(128, 64), # 64차원으로 축소
            nn.ReLU(), # 비선형 활성화 함수 적용
            nn.Linear(64, 12), # 12차원으로 축소
            nn.ReLU(), # 비선형 활성화 함수 적용
            nn.Linear(12, 3), # 최종 3차원 잠재 공간으로 압축
        )
        self.decoder = nn.Sequential( # 디코더 부분 정의
            nn.Linear(3, 12), # 3차원을 다시 12차원으로 확장
            nn.ReLU(), # 비선형 활성화 함수 적용
            nn.Linear(12, 64), # 64차원으로 확장
            nn.ReLU(), # 비선형 활성화 함수 적용
            nn.Linear(64, 128), # 128차원으로 확장
            nn.ReLU(), # 비선형 활성화 함수 적용
            nn.Linear(128, 28*28), # 원래 이미지 크기로 복원
            nn.Sigmoid(), # 출력을 0과 1 사이 값으로 조정
        )

    def forward(self, x): # 순전파 함수
        encoded = self.encoder(x) # 인코더 통과
        decoded = self.decoder(encoded) # 디코더 통과
        return encoded, decoded # 압축된 특징과 복원된 이미지 반환

In [ ]:
# 모델 및 최적화 도구 초기화
autoencoder = Autoencoder().to(DEVICE) # 모델 인스턴스 생성 및 기기 할당
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.005) # 최적화 알고리즘 설정
criterion = nn.MSELoss() # 평균 제곱 오차 손실 함수 설정

In [ ]:
def train(autoencoder, train_loader): # 학습 함수 정의
    autoencoder.train() # 모델을 학습 모드로 전환
    for step, (x, label) in enumerate(train_loader): # 배치 단위로 데이터 순회
        x = x.view(-1, 28*28).to(DEVICE) # 입력을 평탄화하고 기기 할당
        y = x.view(-1, 28*28).to(DEVICE) # 타겟 이미지 설정
        label = label.to(DEVICE) # 레이블 정보 기기 할당

        encoded, decoded = autoencoder(x) # 모델 예측 수행

        loss = criterion(decoded, y) # 손실 계산
        optimizer.zero_grad() # 기울기 초기화
        loss.backward() # 역전파 수행
        optimizer.step() # 가중치 업데이트

In [ ]:
view_data = trainset.data[:5].view(-1, 28*28) # 시각화를 위한 샘플 데이터 선택
view_data = view_data.type(torch.FloatTensor)/255. # 정규화 수행

if not os.path.exists('results'): # 결과 저장 폴더 확인
    os.makedirs('results') # 폴더가 없으면 생성

In [ ]:
# 학습 함수 정의
for epoch in range(1, EPOCH+1): # 에포크 반복
    train(autoencoder, train_loader) # 모델 학습 진행

    # 디코더에서 나온 이미지를 시각화 하기 (두번째 열)
    test_x = view_data.to(DEVICE) # 테스트 데이터 준비
    _, decoded_data = autoencoder(test_x) # 복원 수행

    # 원본과 디코딩 결과 비교해보기
    f, a = plt.subplots(2, 5, figsize=(5, 2)) # 비교 시각화 그래프 생성
    print("[Epoch {}]".format(epoch))
    for i in range(5): # 샘플별 반복
        img = np.reshape(view_data.data.numpy()[i], (28, 28)) # 원본 이미지 복원
        a[0][i].imshow(img, cmap='gray') # 원본 이미지 출력
        a[0][i].set_xticks(()); a[0][i].set_yticks(()) # 축 표시 제거

    for i in range(5): # 복원 이미지 시각화
        img = np.reshape(decoded_data.to("cpu").data.numpy()[i], (28, 28)) # 복원 이미지 복원
        a[1][i].imshow(img, cmap='gray') # 복원 이미지 출력
        a[1][i].set_xticks(()); a[1][i].set_yticks(()) # 축 표시 제거
    
    # 학습 과정 이미지 저장
    plt.savefig(f'results/epoch_{epoch:02d}.png', dpi=300, bbox_inches='tight')
    plt.show()

# 잠재변수 들여다보기

In [ ]:
# 3차원 시각화 (일부 데이터 사용)
view_data = trainset.data[:200].view(-1, 28*28)
view_data = view_data.type(torch.FloatTensor)/255.
test_x = view_data.to(DEVICE)
encoded_data, _ = autoencoder(test_x)
encoded_data = encoded_data.to("cpu")

In [ ]:
CLASSES = {
    0: 'T-shirt/top',
    1: 'Trouser',
    2: 'Pullover',
    3: 'Dress',
    4: 'Coat',
    5: 'Sandal',
    6: 'Shirt',
    7: 'Sneaker',
    8: 'Bag',
    9: 'Ankle boot'
}

# 새로운 figure 생성
plt.clf()  # 이전 plot 초기화
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')  # 3D subplot 생성

X = encoded_data.data[:, 0].numpy()
Y = encoded_data.data[:, 1].numpy()
Z = encoded_data.data[:, 2].numpy()

labels = trainset.targets[:200].numpy()

# 산점도 그리기
scatter = ax.scatter(X, Y, Z, c=labels, cmap='rainbow', s=100, alpha=0.8)

# 텍스트 레이블 추가 (일부만 표시)
for i, (x, y, z, s) in enumerate(zip(X, Y, Z, labels)):
    if i % 20 == 0:  # 20개마다 하나씩만 표시
        name = CLASSES[s]
        color = cm.rainbow(int(255*s/9))
        ax.text(x, y, z, name, color='black', fontsize=10, 
                bbox=dict(facecolor=color, alpha=0.5, edgecolor='black', pad=2))

# 축 레이블 추가
ax.set_xlabel('Latent 1', fontsize=12)
ax.set_ylabel('Latent 2', fontsize=12)
ax.set_zlabel('Latent 3', fontsize=12)

# 축 범위 설정
margin = 0.1  # 여백 추가
x_range = X.max() - X.min()
y_range = Y.max() - Y.min()
z_range = Z.max() - Z.min()

ax.set_xlim(X.min() - margin * x_range, X.max() + margin * x_range)
ax.set_ylim(Y.min() - margin * y_range, Y.max() + margin * y_range)
ax.set_zlim(Z.min() - margin * z_range, Z.max() + margin * z_range)

# 컬러바 추가
cbar = plt.colorbar(scatter)
cbar.set_label('Class', fontsize=12)

# 제목 추가
plt.title('3D Visualization of Latent Space', fontsize=14, pad=20)

# 그리드 추가
ax.grid(True)

# 시점 설정
ax.view_init(elev=20, azim=45)

# 레이아웃 조정
plt.tight_layout()

# 저장 및 표시
plt.savefig('results/latent_space_3d.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n모든 이미지가 'results' 디렉토리에 저장되었습니다.")
print("- 학습 과정 이미지: epoch_01.png ~ epoch_10.png")
print("- 3D 시각화 이미지: latent_space_3d.png")